# 医療機関ホームページURL パイロット（1,000件）

上から順にセルの左にある ▶ を押していくだけです。
各セルは前のセルが終わってから押してください。

所要時間の目安は全部で30〜40分です。

## 1. 準備

プログラム一式を取り込みます。30秒ほどで終わります。

In [ ]:
!git clone -b claude/medical-facility-homepage-urls-v8z8e9 https://github.com/smasuzoe-jpg/search-web.git
%cd search-web
!pip install -q -r pipeline/requirements.txt
print("準備できました")

## 2. スプレッドシートを読み込む

実行すると「このノートブックにGoogleドライブへのアクセスを許可しますか」と聞かれます。
ご自身のアカウントを選んで「許可」を押してください。

読み込むのは リスト① です。②③をやるときは下の URL の部分だけ差し替えます。

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread, csv, os
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

# ← ②③を処理するときはこの1行だけ差し替える
URL = "https://docs.google.com/spreadsheets/d/1ZpqN1nu7q2D-Nl4KatbRtF9tb28uqEm-d5uqGi6Qyfk/edit"

sh = gc.open_by_url(URL)
header, all_rows = None, []
for ws in sh.worksheets():
    vals = ws.get_all_values()
    print(f"シート「{ws.title}」: {len(vals)} 行")
    if not vals:
        continue
    if header is None:
        header = vals[0]
        all_rows.extend(vals)
    else:
        all_rows.extend(vals[1:] if vals[0] == header else vals)

os.makedirs("data", exist_ok=True)
with open("data/list1.csv", "w", encoding="utf-8", newline="") as f:
    csv.writer(f).writerows(all_rows)

print()
print(f"合計 {len(all_rows) - 1} 件を読み込みました")
print("列:", header)

## 3. APIキーを入れる

実行すると入力欄が出ます。Serperの API Key を貼り付けて Enter を押してください。

**画面には表示されず、ノートブックにも保存されません。** 安全に扱われます。

In [ ]:
import os, getpass
os.environ["SERPER_API_KEY"] = getpass.getpass("Serper の API Key を貼り付けて Enter: ")
print("設定しました")

## 4. 下ごしらえ（データの整理）

施設名や住所を整えて、検索用のキーワードを作ります。1分ほどです。
ネットは使わないので、APIの残数は減りません。

In [ ]:
!python3 pipeline/01_prepare.py data/list1.csv > data/work1.csv
!python3 pipeline/head_csv.py data/work1.csv 1000 > data/pilot.csv
print("下ごしらえ完了")

## 5. 検索する

1,000件をまとめて検索します。5分ほどです。
Serperの無料枠2,500件のうち1,000件を使います。

途中で止まっても、もう一度このセルを押せば続きから再開します。

In [ ]:
!python3 pipeline/03_search.py data/pilot.csv data/pilot_cand.jsonl

## 6. 照合する（いちばん時間がかかります）

見つかったサイトを実際に開いて、電話番号・番地・施設名が一致するか確かめます。
**15〜30分かかります。** 実行中はこのタブを閉じないでください。

途中で止まっても、もう一度このセルを押せば続きから再開します。

In [ ]:
!python3 pipeline/04_verify.py data/pilot_cand.jsonl data/pilot_verified.tsv

## 7. 結果を出す

最後に出る `[export]` から始まる集計を、そのままコピーして共有してください。

In [ ]:
!python3 pipeline/05_export.py data/pilot.csv data/pilot_verified.tsv data/pilot

## 8. 「高」判定を20件チェックする

**ここがいちばん大事な作業です。**

「高」と判定されたURLは無条件でHubSpotに反映されます。
下に出る20件について、URLを開いて「本当にその医療機関の公式サイトか」を目視してください。

- 20件中19件以上が正しい → そのまま本番へ進めます
- それ未満 → 判定基準を厳しくします。結果を教えてください

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 60)

df = pd.read_csv("data/pilot_audit.csv")
counts = df["確度"].value_counts()
print(counts.to_string())
print()

high = df[df["確度"] == "高"]
print(f"「高」判定は {len(high)} 件。うち20件を無作為に抜き出します。")
high.sample(min(20, len(high)), random_state=1)[["会社名", "住所", "ウェブサイトURL", "判定根拠"]]

## 9. 結果のファイルを手元に落とす

`pilot_audit.csv` がダウンロードされます。これを共有してください。
APIキーは含まれていません。

In [ ]:
from google.colab import files
files.download("data/pilot_audit.csv")